# Encuadre: recortar, cambiar el tamaño y espejar

**Materiales desarrollados por Matías Barreto, 2026**

**Tecnicatura Superior en Ciencias de Datos e IA, IFTS24**
* **Nomenclatura oficial:** Procesamiento de Imágenes
* **Nombre de trabajo:** Laboratorio de Tecnologías de la Imagen Digital

---

## Objetivo

Decidir qué parte de una imagen conservar y a qué tamaño, con recortes, `cv2.resize` y `cv2.flip`, entendiendo qué información se pierde o se inventa en cada operación.

## Al terminar este cuaderno van a poder:

1. Recortar una región indicando primero las filas y después las columnas.
2. Explicar por qué un recorte de NumPy puede modificar la imagen original.
3. Redimensionar con `cv2.resize` sin confundir el orden entre ancho y alto.
4. Cambiar el tamaño de una imagen conservando su proporción.
5. Comparar métodos de interpolación al agrandar y al achicar.
6. Espejar y rotar en múltiplos de 90 grados con `cv2.flip` y `cv2.rotate`.

## ◈ Microglosario

| Término | Qué es en lenguaje llano |
|---|---|
| **Región de interés** | La parte de la imagen que nos importa para una tarea: una cara, una patente, un objeto. |
| **Vista** | Un recorte de NumPy que no copia los datos: mira la misma memoria que la imagen original. |
| **Redimensionar** | Cambiar la cantidad de filas y columnas de una imagen. |
| **Proporción** | La relación entre el ancho y el alto. Si no se respeta, la imagen se deforma. |
| **Interpolación** | La regla que decide qué valor darle a un píxel nuevo que no existía en la imagen original. |
| **Espejado** | Invertir la imagen de izquierda a derecha o de arriba abajo. |

### ⚙ Configuración del entorno

In [ ]:
from pathlib import Path

import cv2
import numpy as np

from utils import mostrar_imágenes

print(f"OpenCV {cv2.__version__} — NumPy {np.__version__}")

Volvemos a definir `leer_imagen_rgb`, la función de `001_traduccion`, y cargamos una fotografía aérea de dos personas pescando en un bote.

In [ ]:
CARPETA_IMÁGENES = Path("..") / "imagenes"

def leer_imagen_rgb(ruta_de_la_imagen):
    """Lee una imagen con OpenCV, verifica la lectura y la devuelve en RGB."""
    imagen_leída = cv2.imread(filename=str(ruta_de_la_imagen), flags=cv2.IMREAD_COLOR)
    assert imagen_leída is not None, f"No se pudo leer: {ruta_de_la_imagen}"
    imagen_convertida = cv2.cvtColor(src=imagen_leída, code=cv2.COLOR_BGR2RGB)
    return imagen_convertida

imagen_botes = leer_imagen_rgb(CARPETA_IMÁGENES / "small-boats_1.jpg")
forma_botes = imagen_botes.shape
print(f"Imagen cargada: {forma_botes}")

La imagen tiene 795 filas y 1200 columnas. Mirémosla completa antes de decidir qué recortar.

In [ ]:
paneles = [{"imagen": imagen_botes, "título": "small-boats_1.jpg"}]
mostrar_imágenes(paneles, alto_de_figura=6)

Además del bote hay un grupo de nenúfares flotando a la izquierda, sobre el agua oscura. Esa zona va a ser nuestra primera región de interés.

## 1. Recortar

**Analogía.** Recortar una foto impresa con una tijera: lo que queda del lado de afuera no se reduce ni se transforma, simplemente se descarta.

**Bajada tecnológica.** Casi todo sistema de reconocimiento recorta antes de analizar: un lector de patentes recorta la chapa, un reconocedor facial recorta la cara. Trabajar sobre la región de interés ahorra cálculo y evita que el fondo confunda al algoritmo.

En OpenCV no existe una función especial para recortar: se usa el mismo slicing de NumPy que en fundamentos. El orden es siempre **filas primero, columnas después**.

### ✎ Antes de ejecutar

Vamos a recortar desde la fila 500 hasta la 680 y desde la columna 180 hasta la 380. ¿Qué forma va a tener el recorte?

In [ ]:
# Slicing: primero el rango de filas (vertical), después el de columnas (horizontal).
fila_inicial = 500
fila_final = 680
columna_inicial = 180
columna_final = 380

recorte_nenúfares = imagen_botes[fila_inicial:fila_final, columna_inicial:columna_final]

forma_del_recorte = recorte_nenúfares.shape
print(f"Forma del recorte: {forma_del_recorte}")

El recorte tiene 180 filas (680 − 500) y 200 columnas (380 − 180). La cantidad de canales no cambia: seguimos teniendo una imagen en color.

In [ ]:
paneles = [
    {"imagen": imagen_botes, "título": "Imagen completa"},
    {"imagen": recorte_nenúfares, "título": "Recorte: nenúfares"},
]
mostrar_imágenes(paneles)

El recorte muestra los nenúfares con el mismo detalle que tenían en la imagen completa: no se agregó ni se quitó información dentro de la región, solo se descartó lo de alrededor.

### Un recorte no es una copia

Hay una trampa importante. Cuando NumPy recorta, no copia los datos: devuelve una **vista** que apunta a la misma memoria que la imagen original. Lo comprobamos sobre una copia de la imagen, para no arruinar la que vamos a seguir usando.

In [ ]:
imagen_de_prueba = imagen_botes.copy()
recorte_de_prueba = imagen_de_prueba[fila_inicial:fila_final, columna_inicial:columna_final]

# Pintamos de negro únicamente el recorte.
recorte_de_prueba[:, :] = 0

zona_en_la_imagen_completa = imagen_de_prueba[fila_inicial:fila_final, columna_inicial:columna_final]
máximo_en_la_zona = zona_en_la_imagen_completa.max()
print(f"Valor máximo en esa zona de la imagen completa: {máximo_en_la_zona}")

paneles = [{"imagen": imagen_de_prueba, "título": "Imagen de prueba después de pintar el recorte"}]
mostrar_imágenes(paneles, alto_de_figura=5)

Solo pintamos el recorte, pero en la imagen completa aparece un rectángulo negro y el valor máximo de esa zona es 0. El recorte y la imagen comparten memoria. Si quieren modificar un recorte sin tocar el original, tienen que usar `.copy()`, como hicimos al crear `imagen_de_prueba`.

### ✎ Para pensar

- ¿Qué pasaría si intentan recortar desde la fila 700 hasta la 900, en una imagen que tiene 795 filas? Prueben la hipótesis antes de ejecutar.
- ¿En qué situación la vista es una ventaja y no una trampa?

## 2. Cambiar el tamaño

**Analogía.** Una fotocopiadora puede reducir o ampliar. Al reducir, detalles chicos se pierden; al ampliar, la copia se ve más grande, pero no aparece ningún detalle que no estuviera en el original.

**Bajada tecnológica.** Las redes neuronales de clasificación de imágenes exigen un tamaño de entrada fijo, por ejemplo 224 × 224 píxeles. Antes de llegar al modelo, todas las fotos pasan por un `resize`. Lo van a volver a ver en los módulos de redes neuronales.

### ✎ Antes de ejecutar

`cv2.resize` recibe el tamaño nuevo en el parámetro `dsize`. Si escribimos `dsize=(400, 360)`, ¿qué forma esperan: `(400, 360, 3)` o `(360, 400, 3)`?

In [ ]:
# 'resize' cambia la cantidad de píxeles. 'dsize' se escribe como (ancho, alto).
recorte_ampliado = cv2.resize(src=recorte_nenúfares, dsize=(400, 360))

forma_ampliada = recorte_ampliado.shape
print(f"Forma original: {forma_del_recorte} → forma nueva: {forma_ampliada}")

La forma resultante es `(360, 400, 3)`: 360 filas y 400 columnas. `dsize` va en el orden **(ancho, alto)**, el inverso de `.shape`, que va en el orden (alto, ancho). Si se confunden, OpenCV no da error: devuelve una imagen deformada.

Para no deformar, el tamaño nuevo tiene que conservar la proporción entre ancho y alto. Elegimos el ancho que queremos y calculamos el alto que le corresponde.

In [ ]:
alto_original = imagen_botes.shape[0]
ancho_original = imagen_botes.shape[1]
proporción = ancho_original / alto_original

ancho_nuevo = 600
# El alto que conserva la proporción, redondeado a un número entero de píxeles.
alto_nuevo = round(ancho_nuevo / proporción)

botes_proporcionados = cv2.resize(src=imagen_botes, dsize=(ancho_nuevo, alto_nuevo))
botes_deformados = cv2.resize(src=imagen_botes, dsize=(600, 600))

print(f"Proporción original: {proporción:.2f} → tamaño nuevo: {ancho_nuevo} × {alto_nuevo}")

La imagen es aproximadamente 1,5 veces más ancha que alta, así que a un ancho de 600 le corresponde un alto de 397: la cuenta da 397,49…, y `round` la lleva al entero más cercano. Comparamos esa versión con una forzada a 600 × 600.

In [ ]:
paneles = [
    {"imagen": botes_proporcionados, "título": f"{ancho_nuevo} × {alto_nuevo}: proporción conservada"},
    {"imagen": botes_deformados, "título": "600 × 600: proporción forzada"},
]
mostrar_imágenes(paneles, alto_de_figura=5)

En la versión forzada el bote se ve más alto y angosto, y las personas quedan estiradas en sentido vertical. La imagen tiene la cantidad de píxeles que pedimos, pero ya no representa la escena con sus proporciones reales.

### ✎ Para pensar

- Si un modelo exige imágenes de 224 × 224 y nuestras fotos son rectangulares, ¿qué alternativas tenemos además de deformarlas?
- ¿Por qué usamos `round` para calcular el alto nuevo?

## 3. Interpolación: inventar píxeles con criterio

**Analogía.** Si agrandamos un mosaico, aparecen huecos entre las baldosas. Podemos rellenarlos copiando la baldosa más cercana, o mezclando los colores de las baldosas vecinas. Cada regla da un aspecto distinto, pero ninguna recupera un dibujo que no estaba.

**Bajada tecnológica.** Los videojuegos de estilo retro agrandan con el vecino más cercano para conservar los píxeles marcados. Los editores de fotos y las cámaras usan métodos que mezclan vecinos para que el resultado se vea suave. Al achicar, se promedian los píxeles para no introducir ruido.

El parámetro `interpolation` de `cv2.resize` decide la regla:

| Valor | Qué hace | Cuándo conviene |
|---|---|---|
| `cv2.INTER_NEAREST` | Copia el píxel más cercano | Máscaras y etiquetas, donde no se deben mezclar valores |
| `cv2.INTER_LINEAR` | Mezcla los 4 vecinos más cercanos | Valor por defecto; agrandar con rapidez |
| `cv2.INTER_CUBIC` | Mezcla 16 vecinos | Agrandar fotos con más suavidad |
| `cv2.INTER_AREA` | Promedia la zona que se comprime | Achicar fotos |

Primero agrandamos un detalle chico, de 40 × 40 píxeles, ocho veces.

In [ ]:
detalle_de_nenúfar = imagen_botes[560:600, 240:280]
tamaño_ampliado = (320, 320)

ampliado_vecino = cv2.resize(src=detalle_de_nenúfar, dsize=tamaño_ampliado, interpolation=cv2.INTER_NEAREST)
ampliado_lineal = cv2.resize(src=detalle_de_nenúfar, dsize=tamaño_ampliado, interpolation=cv2.INTER_LINEAR)
ampliado_cúbico = cv2.resize(src=detalle_de_nenúfar, dsize=tamaño_ampliado, interpolation=cv2.INTER_CUBIC)

paneles = [
    {"imagen": ampliado_vecino, "título": "INTER_NEAREST"},
    {"imagen": ampliado_lineal, "título": "INTER_LINEAR"},
    {"imagen": ampliado_cúbico, "título": "INTER_CUBIC"},
]
mostrar_imágenes(paneles)

Con el vecino más cercano cada píxel original se convierte en un cuadrado de 8 × 8, y la imagen se ve escalonada. Los métodos lineal y cúbico suavizan esos escalones, pero el resultado se ve borroso. Ninguno de los tres muestra detalles nuevos de la hoja: solo cambia la manera de rellenar lo que falta.

Ahora el camino inverso: achicamos la imagen completa a un octavo de su tamaño con dos métodos.

In [ ]:
factor_de_reducción = 8
ancho_reducido = ancho_original // factor_de_reducción
alto_reducido = alto_original // factor_de_reducción
tamaño_reducido = (ancho_reducido, alto_reducido)

reducido_vecino = cv2.resize(src=imagen_botes, dsize=tamaño_reducido, interpolation=cv2.INTER_NEAREST)
reducido_área = cv2.resize(src=imagen_botes, dsize=tamaño_reducido, interpolation=cv2.INTER_AREA)

paneles = [
    {"imagen": reducido_vecino, "título": f"INTER_NEAREST ({ancho_reducido} × {alto_reducido})"},
    {"imagen": reducido_área, "título": f"INTER_AREA ({ancho_reducido} × {alto_reducido})"},
]
mostrar_imágenes(paneles, alto_de_figura=5)

Al achicar con el vecino más cercano, OpenCV se queda con un píxel cada ocho y descarta el resto. En zonas con mucho detalle, como el pasto y los nenúfares, eso produce puntos sueltos, y el borde del bote queda dentado. `INTER_AREA` promedia cada bloque de píxeles y obtiene una versión chica más pareja y fiel a la escena.

### ✎ Modificación guiada

En la celda anterior, cambien `factor_de_reducción` de 8 a **16** y vuelvan a ejecutarla. Antes, anoten en cuál de los dos métodos esperan que la diferencia se note más. Después prueben con **2**: ¿se sigue viendo la diferencia?

## 4. Espejar y rotar en ángulos rectos

**Analogía.** Mirarse en un espejo invierte izquierda y derecha, pero no cambia la cantidad de cosas que se ven. Girar una hoja un cuarto de vuelta tampoco agrega ni quita nada: solo reorganiza la posición del contenido.

**Bajada tecnológica.** Las cámaras frontales de los celulares muestran la imagen espejada para que se comporte como un espejo. En el entrenamiento de redes neuronales se espejan las fotos para multiplicar los ejemplos disponibles, una técnica llamada aumento de datos.

`cv2.flip` recibe un código numérico que indica el eje del espejado:

| `flipCode` | Efecto | Equivalente en NumPy |
|---|---|---|
| `1` | Espejado horizontal (izquierda ↔ derecha) | `np.fliplr(imagen)` |
| `0` | Espejado vertical (arriba ↔ abajo) | `np.flipud(imagen)` |
| `-1` | Los dos a la vez | `np.flipud(np.fliplr(imagen))` |

In [ ]:
espejado_horizontal = cv2.flip(src=recorte_nenúfares, flipCode=1)
espejado_vertical = cv2.flip(src=recorte_nenúfares, flipCode=0)

# 'rotate' gira en múltiplos de 90 grados, sin perder píxeles.
rotado_90_horario = cv2.rotate(src=recorte_nenúfares, rotateCode=cv2.ROTATE_90_CLOCKWISE)

paneles = [
    {"imagen": recorte_nenúfares, "título": "Recorte original"},
    {"imagen": espejado_horizontal, "título": "flipCode=1"},
    {"imagen": espejado_vertical, "título": "flipCode=0"},
    {"imagen": rotado_90_horario, "título": "ROTATE_90_CLOCKWISE"},
]
mostrar_imágenes(paneles)

El espejado horizontal lleva los nenúfares de un lado al otro; el vertical los invierte de arriba abajo. La rotación de 90 grados también reorganiza el contenido, pero además cambia la forma: lo que eran filas pasan a ser columnas.

### Traducción: esto ya lo vimos

En fundamentos espejaron con `np.fliplr`. Comprobemos en números que `cv2.flip` produce exactamente el mismo resultado, y midamos la forma de la imagen rotada.

In [ ]:
espejado_con_numpy = np.fliplr(m=recorte_nenúfares)
coinciden = np.array_equal(a1=espejado_horizontal, a2=espejado_con_numpy)
print(f"¿cv2.flip con flipCode=1 coincide con np.fliplr? {coinciden}")

forma_rotada = rotado_90_horario.shape
print(f"Forma antes de rotar: {forma_del_recorte} | después de rotar: {forma_rotada}")

Los dos espejados son idénticos valor por valor: es la misma operación con otro nombre. La rotación pasó de `(180, 200, 3)` a `(200, 180, 3)`. Las rotaciones en ángulos que no son múltiplos de 90 grados, en cambio, sí necesitan interpolar y rellenar esquinas: las vamos a ver en el nivel fundamental, junto con la corrección de perspectiva.

### ✎ Para pensar

- Si espejan horizontalmente dos veces seguidas, ¿qué imagen obtienen? ¿Y si rotan 90 grados cuatro veces?
- Para aumentar datos de entrenamiento de un clasificador de gatos, ¿tiene sentido espejar horizontalmente? ¿Y verticalmente?

### ✎ Completado

Muchas plataformas piden una miniatura cuadrada. Completen el código para recortar el cuadrado más grande posible en el centro de `imagen_botes` y llevarlo a 256 × 256 píxeles.

In [ ]:
# ─── Completen los pasos faltantes ───────────────────────────────────────────
#
# TODO 1: guarden en 'lado' el menor valor entre alto_original y ancho_original.
# TODO 2: calculen 'fila_desde' y 'columna_desde' para que el cuadrado quede centrado.
# TODO 3: recorten el cuadrado con slicing y redimensiónenlo con dsize=(256, 256).
# TODO 4: muestren el resultado con mostrar_imágenes.
#
# lado = min(alto_original, ancho_original)
# fila_desde = (alto_original - lado) // 2
# columna_desde = ...
# cuadrado_central = imagen_botes[..., ...]
# miniatura = cv2.resize(src=..., dsize=(256, 256), interpolation=...)
#
# ─────────────────────────────────────────────────────────────────────────────

Una vez que funcione, piensen qué método de interpolación eligieron y por qué: ¿están agrandando o achicando?

## Actividad de producción

In [ ]:
# ─── Espacio de práctica ─────────────────────────────────────────────────────
#
# Consigna: preparen una carta para un catálogo.
#   1. Carguen cards.jpg con leer_imagen_rgb.
#   2. Recorten una sola carta de la imagen.
#   3. Llévenla a 300 píxeles de alto conservando su proporción.
#   4. Guárdenla como PNG en la carpeta 'salidas' (recuerden pasarla a BGR).
#   5. Escriban en una celda de texto qué interpolación usaron y qué
#      información de la foto original quedó afuera del resultado.
#
# ─────────────────────────────────────────────────────────────────────────────

## Cierre

Recortar, redimensionar y espejar cambian el encuadre de una imagen. Ninguna de estas operaciones crea información nueva sobre la escena: descartan, reorganizan o rellenan con una regla.

| Operación | Herramienta | Qué pasa con la información |
|---|---|---|
| Recortar | Slicing `[filas, columnas]` | Se descarta lo que queda afuera; el recorte es una vista |
| Achicar | `cv2.resize` con `INTER_AREA` | Se promedian detalles y se pierden los más finos |
| Agrandar | `cv2.resize` con `INTER_LINEAR` o `INTER_CUBIC` | Se inventan píxeles intermedios, sin detalle nuevo |
| Espejar | `cv2.flip` | Se reorganiza sin pérdida |
| Rotar 90 grados | `cv2.rotate` | Se reorganiza sin pérdida; se intercambian alto y ancho |

**Limitaciones de lo que vimos.** Solo trabajamos con rotaciones en ángulos rectos. Las rotaciones en cualquier ángulo y la corrección de perspectiva necesitan matrices de transformación, y quedan para el nivel fundamental. Además, las coordenadas del recorte las elegimos mirando la imagen: un sistema real tendría que encontrar la región de interés de manera automática.

En el próximo cuaderno, `005_resolucion`, vamos a mirar la reducción de información desde otro ángulo: cuántas muestras y cuántos niveles de intensidad necesita una imagen para seguir siendo útil.